In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, scipy.stats as stats, torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F
from torch.optim import Adam
from scipy.stats import spearmanr


### helpers

In [2]:
# normalize a vector from -.4, .4 to be between -1 and 1
def normalize_vector(vec): return vec/.4

### Generate data

In [3]:
data = pd.read_csv('../../data/trials_patients.csv')[:120][::2].reset_index(drop=True)

# convert stim to be between -1 and 1
data['stim'] = normalize_vector(data['stim_pos'])

# map condition to div_pos
condition_to_div_pos = {'baseline': 0, 'curv_comp': -0.5, 'flat_comp': 0.5}
data['div_pos'] = data['condition'].map(condition_to_div_pos)

# creater relative_stim = stim - div_pos
data['relative_stim'] = data['stim'] - data['div_pos']

# convert shape col to binary (0 for curv, 1 for flat)
data['class'] = data['shape'].apply(lambda x: 0 if x == 'curv' else 1)

data[['blockN', 'condition', 'div_pos', 'stim', 'relative_stim', 'class']]

,blockN,condition,div_pos,stim,relative_stim,class
0,1,baseline,0.0,-0.95,-0.95,0
1,1,baseline,0.0,-0.85,-0.85,0
2,1,baseline,0.0,-0.75,-0.75,0
3,1,baseline,0.0,-0.65,-0.65,0
4,1,baseline,0.0,-0.55,-0.55,0
5,1,baseline,0.0,-0.45,-0.45,0
6,1,baseline,0.0,-0.35,-0.35,0
7,1,baseline,0.0,-0.25,-0.25,0
8,1,baseline,0.0,-0.15,-0.15,0
9,1,baseline,0.0,-0.05,-0.05,0


### Organize data

In [4]:
# build tensors
X = data[['div_pos', 'stim_pos']].values.astype(np.float32)
y = data['class'].values.astype(np.float32)

# train/val split
rng = np.random.default_rng(0)
idx = rng.permutation(len(data))
split = int(0.8 * len(data))
train_idx, val_idx = idx[:split], idx[split:]

# convert to torch tensors
X_train = torch.tensor(X[train_idx])
y_train = torch.tensor(y[train_idx]).unsqueeze(1)
X_val = torch.tensor(X[val_idx])
y_val = torch.tensor(y[val_idx]).unsqueeze(1)

# build datasets/loaders
train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)

# sanity check
train_loader, val_loader

(<torch.utils.data.dataloader.DataLoader at 0x7fe1c9c2c650>,
 <torch.utils.data.dataloader.DataLoader at 0x7fe064ecb710>)

### Define NN architecture

In [5]:
class SimpleNet(nn.Module):
    # small feedforward binary classifier
    def __init__(self, in_dim=2, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        # forward pass
        return self.net(x)

# model/criterion/optimizer
model = SimpleNet()
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=1e-3)

def run_epoch(loader, train=True):
    # one epoch over loader
    if train:
        model.train()
    else:
        model.eval()

    losses = []
    correct = 0
    total = 0

    for xb, yb in loader:
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        losses.append(loss.item())

        # compute accuracy
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == yb).sum().item()
        total += yb.numel()

    return float(np.mean(losses)), correct / total

# training loop
for epoch in range(20):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    if epoch % 5 == 0:
        print(f"epoch {epoch:02d} | train loss {train_loss:.4f}, acc {train_acc:.3f} | val loss {val_loss:.4f}, acc {val_acc:.3f}")

epoch 00 | train loss 0.7033, acc 0.354 | val loss 0.7023, acc 0.500
epoch 05 | train loss 0.6997, acc 0.292 | val loss 0.7014, acc 0.333
epoch 10 | train loss 0.6964, acc 0.438 | val loss 0.7009, acc 0.250
epoch 15 | train loss 0.6938, acc 0.354 | val loss 0.7009, acc 0.250


### Train

### Test

### Plot behavior

### Plot neural